In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="cost",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.silver.cost where effective_from_date = '2024-01-01' limit 1

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

def upsert_unit_cost(microBatchDF, batchId):
        
    # Read the cost table for the current batch
    cost_df = microBatchDF.select(
        "deployment_name", "upc_code", "unit_cost", "effective_from_date", "effective_to_date"
    )

    (DeltaTable.forName(spark, "fq_dev_catalog.gold.sales_item_wise").alias("target")
    .merge(
        cost_df.alias("source"),
        "target.deployment_name = source.deployment_name AND "
        "target.item_code = source.upc_code AND "
        "target.business_date >= source.effective_from_date AND "
        "target.business_date <= source.effective_to_date"
    )
    .whenMatchedUpdate(set={"unit_cost": "source.unit_cost"})
    .execute())

(
    spark.readStream
    .table("fq_dev_catalog.silver.cost")
    .writeStream
    .foreachBatch(upsert_unit_cost)
    .option("checkpointLocation", f"{checkpoint}/{source}/{domain}/streaming/checkpoint_gold_unit_cost")
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [0]:
%sql
SELECT 
  count(distinct deployment_name) as deployment_name_cardinality,
  count(distinct business_date) as business_date_cardinality,
  count(distinct item_code) as item_code_cardinality,
  count(distinct unit_cost) as unit_cost_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.gold.sales_item_wise;

In [0]:
%sql
OPTIMIZE fq_dev_catalog.gold.sales_item_wise ZORDER BY (item_name, item_code, business_date); -- high cardinality non-partitioned column first

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_item_wise where super_category != 'Combo' and business_date = '2024-01-01' and gross_amount > 0 limit 10

In [0]:
for query in spark.streams.active:
    query.stop()